# 01b — Extract SoilGrids (ISRIC)
**Data source:** [SoilGrids REST API](https://rest.isric.org/) — 6 soil properties at 0-5cm depth

**Properties:** pH (H2O), Clay %, Sand %, Silt %, Organic Carbon Density, Cation Exchange Capacity

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00

**Output:** `soilgrids.parquet` (one row per unique station, 6 soil feature columns)

**Estimated time:** ~30-60 min for ~170 stations (6 properties each, rate limited)

> Enable Internet in Kaggle settings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, time, requests, logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01b_soilgrids')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

INPUT_DIR  = '/kaggle/input/ey-water-quality-nb00'
OUTPUT_DIR = '/kaggle/working'

LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

SOIL_PROPERTIES = ['phh2o', 'clay', 'sand', 'silt', 'ocd', 'cec']

log.info(f'Will extract {len(SOIL_PROPERTIES)} soil properties per station')

In [ ]:
train_base = pd.read_parquet(f'{INPUT_DIR}/train_base.parquet')
val_base   = pd.read_parquet(f'{INPUT_DIR}/val_base.parquet')

all_data = pd.concat([train_base, val_base], ignore_index=True)
unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()

log.info(f'Train: {train_base.shape}, Val: {val_base.shape}')
log.info(f'Unique stations: {len(unique_stations)}')

---
## Extraction

In [ ]:
def fetch_soilgrids_single(lat, lon, prop, retries=3):
    """Fetch one soil property for one location."""
    url = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
    for attempt in range(retries):
        try:
            r = requests.get(url, params={
                'lat': lat, 'lon': lon,
                'property': prop, 'depth': '0-5cm', 'value': 'mean'
            }, timeout=30)
            r.raise_for_status()
            val = r.json()['properties']['layers'][0]['depths'][0]['values']['mean']
            return float(val) if val is not None else np.nan
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return np.nan

In [ ]:
total = len(unique_stations)
results = []
start_time = time.time()

log.info(f'Starting SoilGrids extraction for {total} stations x {len(SOIL_PROPERTIES)} properties...')

for idx, row in unique_stations.iterrows():
    lat, lon = row[LAT_COL], row[LON_COL]
    station = row[STATION_COL]
    
    record = {STATION_COL: station, LAT_COL: lat, LON_COL: lon}
    prop_status = []
    
    for prop in SOIL_PROPERTIES:
        val = fetch_soilgrids_single(lat, lon, prop)
        record[f'soil_{prop}'] = val
        prop_status.append('OK' if not np.isnan(val) else 'FAIL')
        time.sleep(0.5)  # rate limit
    
    results.append(record)
    done = len(results)
    
    # Log every station (since each takes ~3-5 sec)
    elapsed = time.time() - start_time
    rate = done / elapsed if elapsed > 0 else 0
    eta = (total - done) / rate if rate > 0 else 0
    status_str = ' '.join(f'{p}:{s}' for p, s in zip(SOIL_PROPERTIES, prop_status))
    log.info(f'  [{done:3d}/{total}] {done/total*100:5.1f}% | '
             f'ETA {eta/60:.1f}m | {station[:25]} | {status_str}')

elapsed_total = time.time() - start_time
log.info(f'DONE in {elapsed_total/60:.1f} min')

In [ ]:
soil_df = pd.DataFrame(results)

# Summary
log.info(f'Output shape: {soil_df.shape}')
for prop in SOIL_PROPERTIES:
    col = f'soil_{prop}'
    nulls = soil_df[col].isnull().sum()
    log.info(f'  {col}: {nulls} nulls, '
             f'range [{soil_df[col].min():.1f}, {soil_df[col].max():.1f}]')

display(soil_df.describe())

---
## Figure: Soil Properties Map Grid

In [ ]:
n_props = len(SOIL_PROPERTIES)
ncols = 3
nrows = (n_props + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()

for i, prop in enumerate(SOIL_PROPERTIES):
    ax = axes[i]
    col = f'soil_{prop}'
    valid = soil_df[col].notna()
    
    if valid.sum() > 0:
        sc = ax.scatter(soil_df.loc[valid, LON_COL], soil_df.loc[valid, LAT_COL],
                       c=soil_df.loc[valid, col], cmap='YlOrRd', s=50,
                       edgecolors='gray', linewidths=0.3)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    
    if (~valid).any():
        ax.scatter(soil_df.loc[~valid, LON_COL], soil_df.loc[~valid, LAT_COL],
                  c='red', marker='x', s=60, linewidths=1.5)
    
    n_ok = valid.sum()
    ax.set_title(f'{prop} ({n_ok}/{len(soil_df)} OK)', fontsize=11)
    ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
    ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
    ax.grid(True, alpha=0.2)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('SoilGrids Properties per Station (0-5cm depth)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01b_soilgrids_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/soilgrids.parquet'
soil_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024

log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(soil_df)} rows, {len(SOIL_PROPERTIES)} properties)')
print(f'\n=== DONE ===')
print(f'Output: soilgrids.parquet')
print(f'Rows: {len(soil_df)}, Cols: {soil_df.columns.tolist()}')
print(f'Next: add this notebook output as dataset input for 01e')